# Tutorial: explain brain IDPs with HomoloMap cell-type maps

This tutorial shows the intended HomoloMap workflow: **the user supplies regional brain imaging-derived phenotypes (IDPs)**, while HomoloMap supplies homologous cell-type maps in the same atlas. We will align regions, run spatial spin tests, estimate the joint contribution of all cell types, and optionally calculate individual cell-type SHAP contributions.

The structure follows the progressive, executable style used by the [BrainSpace null-model tutorial](https://brainspace.readthedocs.io/en/latest/python_doc/auto_examples/plot_tutorial3.html). The cell-type source and mapping provenance are described in the project [README](../README.md).

## 1. Installation and imports

Install the repository from its root directory. The `all` extra adds SHAP and surface-plotting dependencies.

In [ ]:
# Run once in a fresh environment:
# %pip install -e "..[all]"

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from HomoloMap.stats import SpinTest
from HomoloMap.utils import (
    run_spin_correlations,
    run_cumulative_models,
    run_explanation_analysis,
)

SEED = 42
N_SPINS = 100  # quick tutorial; use >=1,000 for a real analysis
rng = np.random.default_rng(SEED)

## 2. Load a released cell-type map

The primary analysis uses mapped, reclosed ratios. Choose one biological resolution in advance: 23 subclasses for a compact primary analysis or 71 clusters for finer secondary analysis. Do not combine both resolutions into one multiple-testing family.

In [ ]:
def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for root in candidates:
        if (root / 'data' / 'maps' / 'BN').exists():
            return root
    raise FileNotFoundError('Run this notebook from the repository or tutorials directory.')

ROOT = find_repo_root()
CELLTYPE_LEVEL = 'subclass'  # change to 'cluster' for the 71-cluster map
filename = {
    'subclass': 'ctype_ratio_BN_23_subclass.csv',
    'cluster': 'ctype_ratio_BN_71_cluster.csv',
}[CELLTYPE_LEVEL]

X = pd.read_csv(ROOT / 'data' / 'maps' / 'BN' / filename, index_col=0)
X.index = X.index.astype(int)
print('Cell-type predictors:', X.shape)
X.iloc[:3, :5]

## 3. Load your brain IDPs

Your CSV should contain one BN region per row and one IDP per column. Its index must use the BN labels `1, 3, ..., 209`. Set `IDP_PATH` to your file. The fallback below creates two deterministic toy maps only so that every cell in the tutorial can be executed; **toy results have no scientific meaning**.

In [ ]:
IDP_PATH = None  # e.g. ROOT / 'my_data' / 'brain_idps_bn.csv'

if IDP_PATH is None:
    position = np.linspace(-1, 1, len(X))
    Y = pd.DataFrame(
        {
            'toy_IDP_1': position + rng.normal(0, 0.35, len(X)),
            'toy_IDP_2': np.sin(np.pi * position) + rng.normal(0, 0.35, len(X)),
        },
        index=X.index,
    )
    print('Using synthetic IDPs for an execution check only.')
else:
    Y = pd.read_csv(IDP_PATH, index_col=0)
    Y.index = Y.index.astype(int)

print('Brain IDPs:', Y.shape)
Y.head()

## 4. Align regions and run quality control

Alignment is label-based, never row-position based. Record the retained labels and investigate missing regions before analysis. Ratio maps must be finite, non-negative, and closed within each region.

In [ ]:
if not X.index.is_unique or not Y.index.is_unique:
    raise ValueError('ROI labels must be unique.')

shared = X.index.intersection(Y.index, sort=False)
X_aligned = X.loc[shared].astype(float)
Y_aligned = Y.loc[shared].apply(pd.to_numeric, errors='coerce')
valid = X_aligned.notna().all(axis=1) & Y_aligned.notna().all(axis=1)
X_aligned, Y_aligned = X_aligned.loc[valid], Y_aligned.loc[valid]

assert np.isfinite(X_aligned.to_numpy()).all()
assert (X_aligned.to_numpy() >= 0).all()
assert np.allclose(X_aligned.sum(axis=1), 1.0)

print(f'Shared valid regions: {len(X_aligned)}')
print(f'Cell types: {X_aligned.shape[1]}; IDPs: {Y_aligned.shape[1]}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
sns.heatmap(X_aligned.T, cmap='mako', xticklabels=False, ax=ax)
ax.set(xlabel='BN regions', ylabel='Cell types', title='Released BN cell-type composition')
fig.tight_layout()

## 5. Individual spatial associations: spin tests

Ordinary parametric p-values are inappropriate when cortical maps are spatially autocorrelated. HomoloMap rotates parcel centroids on the spherical surface, following the logic of spatial spin null models. The adjusted p-value columns use Benjamini–Hochberg correction separately within each IDP across the selected cell-type resolution. A complete BN atlas is required for the current spin implementation.

In [ ]:
expected_bn = np.arange(1, 210, 2)
if not np.array_equal(X_aligned.index.to_numpy(), expected_bn):
    raise ValueError('This tutorial spin test requires all 105 BN cortical labels.')

spinner = SpinTest(atlas='BN', n_spins=N_SPINS, method='Alexander-Bloch', seed=SEED)
spin_results = run_spin_correlations(
    X_aligned, Y_aligned, spinner, metric='pearsonr',
    FDR='fdr_bh', n_jobs=1, composition_transform='none',
)
spin_results.head()

In [ ]:
idp = Y_aligned.columns[0]
r_col = f'{idp}_ratio_spin_r'
q_col = f'{idp}_ratio_spin_p_adj'
display(spin_results[[r_col, q_col]].sort_values(q_col).head(10))

## 6. Total cell-type contribution

The cumulative model asks how much regional variation in each IDP is captured jointly by all selected cell-type maps. Its spatial permutation p-value tests the complete predictor set against rotated outcomes; it is not the sum of individual spin-test effects.

In [ ]:
total_models = run_cumulative_models(
    X_aligned, Y_aligned, spinner, mode='linear',
    n_spins=N_SPINS, FDR='fdr_bh', n_jobs=1,
    composition_transform='none',
)
total_models

## 7. Individual cell-type contributions with SHAP

SHAP partitions a fitted model's predictions among its input features. `total_contribution` summarizes the model-wide attribution magnitude, whereas `individual_ctype_contribution` reports each cell type's relative share. These are model-dependence measures, not causal effects. This cell requires the `explain` or `all` installation extra.

In [ ]:
RUN_SHAP = False  # set True after installing HomoloMap[explain]

if RUN_SHAP:
    explanations = run_explanation_analysis(
        X_aligned, Y_aligned, method='shap', mode='linear',
        n_jobs=1, random_state=SEED, composition_transform='none',
    )
    first = explanations[Y_aligned.columns[0]]
    print('Total contribution:', first['total_contribution'])
    display(first['individual_ctype_contribution'].head(10))

## 8. CLR sensitivity analysis

Cell-type ratios are compositional. Keep the mapped ratio analysis as the primary, directly interpretable result, then repeat it with a centered log-ratio (CLR) transform to test whether conclusions depend on closure. CLR effects describe relative log-contrasts rather than absolute abundance.

In [ ]:
clr_spin_results = run_spin_correlations(
    X_aligned, Y_aligned, spinner, metric='pearsonr',
    FDR='fdr_bh', n_jobs=1, composition_transform='clr',
    composition_params={'zero_method': 'multiplicative'},
)

comparison = pd.DataFrame({
    'ratio_r': spin_results[r_col],
    'clr_r': clr_spin_results[r_col],
})
comparison.corr()

## 9. Save a reproducible result bundle

Always save the aligned inputs together with the analysis outputs and parameters. For a scientific run, increase `N_SPINS`, predefine the multiple-testing families, and report atlas/hemisphere, mapping coverage, unresolved source mass, seed, model, and transform.

In [ ]:
OUTPUT = ROOT / 'tutorial_outputs'
OUTPUT.mkdir(exist_ok=True)
X_aligned.to_csv(OUTPUT / 'aligned_celltype_predictors.csv')
Y_aligned.to_csv(OUTPUT / 'aligned_brain_idps.csv')
spin_results.to_csv(OUTPUT / 'spin_results_ratio.csv')
clr_spin_results.to_csv(OUTPUT / 'spin_results_clr_sensitivity.csv')
total_models.to_csv(OUTPUT / 'total_models.csv')
print('Saved to', OUTPUT.resolve())

## References

- Chen et al. (2023), *Single-cell spatial transcriptome reveals cell-type organization in the macaque cortex*. [Cell](https://doi.org/10.1016/j.cell.2023.06.009).
- Fan et al. (2016), *The Human Brainnetome Atlas*. [Cerebral Cortex](https://doi.org/10.1093/cercor/bhw157).
- Alexander-Bloch et al. (2018), *On testing for spatial correspondence between maps of human brain structure and function*. [NeuroImage](https://doi.org/10.1016/j.neuroimage.2018.05.070).
- [BrainSpace null-model tutorial](https://brainspace.readthedocs.io/en/latest/python_doc/auto_examples/plot_tutorial3.html).